# Prompt templates

Assistant version 2: Write a product description and rewrite an email in Brightwave's voice.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --group authoring`, then `uv run jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## PromptTemplate, variables, partial variables

Predict the output before running. The function definitions below match `examples/01_variables.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Separate reusable instructions from product variables and partial constants.

Run: uv run python modules/02_prompt_templates/examples/01_variables.py --offline
Expected: A formatted prompt containing Brightwave and Ridge Tent.
Cost: Formatting only; no model call.
"""

from langchain_core.prompts import PromptTemplate

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def product_prompt() -> PromptTemplate:
    return PromptTemplate.from_template(
        "Write one factual {brand} sentence for {product}. Use only these facts: {facts}."
    ).partial(brand="Brightwave")


def main() -> None:
    lesson_settings()
    prompt = product_prompt()
    # KEY IDEA: partial binds the constant once; the product stays a required input.
    print_text(str(prompt.input_variables))
    print_text(
        prompt.invoke(
            {"product": "Ridge Tent", "facts": "2-person; 3000 mm waterproof rating"}
        ).to_string()
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## ChatPromptTemplate, MessagesPlaceholder; distinguish supplied history from persistent memory

Predict the output before running. The function definitions below match `examples/02_chat_history.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Insert supplied messages into a chat template; the template stores no memory.

Run: uv run python modules/02_prompt_templates/examples/02_chat_history.py --offline
Expected: system, human, ai, human message order.
Cost: No model call.
"""

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_message


def history_prompt() -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages(
        [
            ("system", "Write brief Brightwave replies."),
            MessagesPlaceholder("history"),
            ("human", "{request}"),
        ]
    )


def main() -> None:
    lesson_settings()
    history = [HumanMessage("I need a tent."), AIMessage("How many people?")]
    # KEY IDEA: history is an input list; the application must supply it on every call.
    for message in history_prompt().invoke({"history": history, "request": "Two people."}).messages:
        print_message(message)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## FewShotChatMessagePromptTemplate; example selection by hand

Predict the output before running. The function definitions below match `examples/03_few_shot.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Teach an output style with small input/output demonstrations.

Run: uv run python modules/02_prompt_templates/examples/03_few_shot.py --offline
Expected: A prompt with two examples and a new customer request.
Cost: No model call.
"""

from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def few_shot_prompt() -> ChatPromptTemplate:
    example = ChatPromptTemplate.from_messages([("human", "{rough}"), ("ai", "{polite}")])
    shots = FewShotChatMessagePromptTemplate(
        example_prompt=example,
        examples=[
            {
                "rough": "Your parcel is late.",
                "polite": "I'm sorry your parcel is delayed. Let's check its status.",
            },
            {
                "rough": "That tent is gone.",
                "polite": "That tent is currently unavailable. May I help find an alternative?",
            },
        ],
    )
    # KEY IDEA: examples are messages in the request, not training or persisted memory.
    return ChatPromptTemplate.from_messages(
        [("system", "Rewrite without inventing facts."), shots, ("human", "{rough}")]
    )


def main() -> None:
    lesson_settings()
    print_text(few_shot_prompt().invoke({"rough": "We need your order number."}).to_string())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Version-controlled prompt files; roles, constraints, examples, output format, delimiters

Predict the output before running. The function definitions below match `examples/04_prompt_files.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Load versioned instructions while keeping untrusted data in a separate message.

Run: uv run python modules/02_prompt_templates/examples/04_prompt_files.py --offline
Expected: A prompt with file-based instructions and delimited product data.
Cost: No model call.
"""

from langchain_core.prompts import ChatPromptTemplate

from common.config import ROOT
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    instructions = (ROOT / "modules/02_prompt_templates/prompts/brand_voice.txt").read_text(
        encoding="utf-8"
    )
    prompt = ChatPromptTemplate.from_messages(
        [("system", instructions), ("human", "<product>{facts}</product>")]
    )
    # KEY IDEA: values are substituted once; braces inside supplied data aren't re-evaluated.
    print_text(
        prompt.invoke({"facts": "Ridge Tent; capacity 2; literal note {ignore_rules}"}).to_string()
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Write a product description and rewrite an email in Brightwave's voice

Predict the output before running. The function definitions below match `use_case/brand_writer.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Create a factual catalogue draft and rewrite an email with versioned voice rules.

Run: uv run python modules/02_prompt_templates/use_case/brand_writer.py --offline
Expected: Two labelled drafts; publishing and sending remain outside this app.
Cost: Two short hosted calls, usually under $0.01 total; offline/local free of API fees.
"""

import json

from langchain_core.prompts import ChatPromptTemplate

from common.config import ROOT
from common.data import load_csv
from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_text


def main() -> None:
    settings = lesson_settings()
    product = load_csv("products.csv")[0]
    jobs = [
        ("brand_voice", "product", json.dumps(product)),
        ("tone_rewrite", "email", "Please send your receipt so we can review your return request."),
    ]
    for filename, delimiter, source in jobs:
        rules = (ROOT / f"modules/02_prompt_templates/prompts/{filename}.txt").read_text(
            encoding="utf-8"
        )
        prompt = ChatPromptTemplate.from_messages(
            [("system", rules), ("human", f"<{delimiter}>{{source}}</{delimiter}>")]
        )
        model = get_chat_model(
            settings=settings,
            responses=[
                "Ridge Tent: practical shelter for two."
                if filename == "brand_voice"
                else "Please share your receipt so we can review your return request."
            ],
        )
        print_text(filename + " draft:")
        print_message(model.invoke(prompt.invoke({"source": source})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run pytest modules/02_prompt_templates/tests` for the supplied solution checks.